In [ ]:
# default_exp paper.plsr.learning_curve

# 3.1. Learning Curve (PLSR)

> Computing the learning curve of PLSR for the prediction of exchangeable potassium (K ex.), with increasing number of training examples and using all Soil Taxonomy Orders. 

In our experiment, the model was given the objective to achieve a Mean Squared Error (MSE) of `10-2` on the training set and was free to scale up its capacity to do so, regardless of the risk of overfitting.

In [ ]:
if 'google.colab' in str(get_ipython()):
    from google.colab import drive
    drive.mount('/content/drive',  force_remount=False)
    !pip install mirzai
else:
    %load_ext autoreload
    %autoreload 2

In [ ]:
# mirzai utilities
from mirzai.data.loading import load_kssl
from mirzai.data.selection import (select_y, select_tax_order, select_X)
from mirzai.data.transform import (log_transform_y, SNV, TakeDerivative,
                                   DropSpectralRegions, CO2_REGION)

from fastcore.transform import compose

# Python utilities
from collections import OrderedDict
from tqdm.auto import tqdm
from pathlib import Path

# Data science stack
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.cross_decomposition import PLSRegression
from sklearn.metrics import mean_squared_error

import warnings
warnings.filterwarnings('ignore')

## Load and transform

In [ ]:
src_dir = '../_data'
fnames = ['spectra-features.npy', 'spectra-wavenumbers.npy', 
          'depth-order.npy', 'target.npy', 
          'tax-order-lu.pkl', 'spectra-id.npy']

X, X_names, depth_order, y, tax_lookup, X_id = load_kssl(src_dir, fnames=fnames)

data = X, y, X_id, depth_order

transforms = [select_y, select_tax_order, select_X, log_transform_y]
X, y, X_id, depth_order = compose(*transforms)(data)

In [ ]:
print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')
print(f'Wavenumbers:\n {X_names}')
print(f'depth_order (first 3 rows):\n {depth_order[:3, :]}')
print(f'Taxonomic order lookup:\n {tax_lookup}')

X shape: (40132, 1764)
y shape: (40132,)
Wavenumbers:
 [3999 3997 3995 ...  603  601  599]
depth_order (first 3 rows):
 [[43.  2.]
 [ 0.  0.]
 [ 0.  1.]]
Taxonomic order lookup:
 {'alfisols': 0, 'mollisols': 1, 'inceptisols': 2, 'entisols': 3, 'spodosols': 4, 'undefined': 5, 'ultisols': 6, 'andisols': 7, 'histosols': 8, 'oxisols': 9, 'vertisols': 10, 'aridisols': 11, 'gelisols': 12}


## Experiment

### Setup

In [ ]:
# Metrics to gather
history = OrderedDict({'nb_samples': [], 'train_score': [], 'valid_score': [], 'nb_components': []})

# Dataset subset sizes
training_size = [ 100, 500, 1000, 2500, 5000, 10000, 20000, 30000, X.shape[0]]

target_mse = 0.01 # Mean Square Error to target

# Define PLSR components range according to dataset size
ranger = lambda s: range(2, 500, 10) if s < 2500 else range(200, 1600, 200)

early_stop = 1e-4 # When no further improvement of training MSE threshold

random_seed = 42
test_size = 0.2 # 20% kept as validation set

### Run

In [ ]:
for size in tqdm(training_size):
    print(100*'-')
    print('# of samples: {}'.format(size))

    idx = np.random.choice(len(X), size, replace=False)
    X_train, X_valid, y_train, y_valid = train_test_split(X[idx, :], 
                                                          y[idx], 
                                                          test_size=test_size, 
                                                          random_state=random_seed)

    pls_range = ranger(size)
    last_score = 999

    for cpts in pls_range:
        pipe = Pipeline([('snv', SNV()),
                         ('derivative', TakeDerivative(window_length=11, polyorder=1)), 
                         ('dropper', DropSpectralRegions(X_names, regions=CO2_REGION)),
                         ('model', PLSRegression(n_components=cpts))])

        pipe.fit(X_train, y_train)
    
        train_score = mean_squared_error(pipe.predict(X_train), y_train)
    
        print("# of pls cpts: ", cpts)
        print("Train score: ", train_score)
        print("Last progress: ", last_score - train_score)
        if last_score - train_score <= early_stop:
            print("Early stopping!")
            break
        
        if train_score <= target_mse:
            print("Reached the target!")
            break

        last_score = train_score
    
    valid_score = mean_squared_error(pipe.predict(X_valid), y_valid)
    
    # Store scores and others in history
    print("Valid_score: ", valid_score)
    history['nb_samples'].append(len(X_train))
    history['train_score'].append(train_score)
    history['valid_score'].append(valid_score)
    history['nb_components'].append(cpts)

  0%|          | 0/9 [00:00<?, ?it/s]

----------------------------------------------------------------------------------------------------
# of samples: 100
# of pls cpts:  2
Train score:  0.12374779826661646
Last progress:  998.8762522017333
# of pls cpts:  12
Train score:  0.025001159515724118
Last progress:  0.09874663875089235
# of pls cpts:  22
Train score:  0.005237744236828559
Last progress:  0.01976341527889556
Reached the target!
Valid_score:  0.19359226189297302
----------------------------------------------------------------------------------------------------
# of samples: 500
# of pls cpts:  2
Train score:  0.09366431779020098
Last progress:  998.9063356822098
# of pls cpts:  12
Train score:  0.04830604892354282
Last progress:  0.04535826886665816
# of pls cpts:  22
Train score:  0.034660901361959345
Last progress:  0.013645147561583477
# of pls cpts:  32
Train score:  0.025297937205260187
Last progress:  0.009362964156699159
# of pls cpts:  42
Train score:  0.019601970088354553
Last progress:  0.0056959671169

### Save results

In [ ]:
dest_dir = 'name_of_your_destination_directory'

with open(Path(dest_dir)/'history_pls_learning_curve.pickle', 'wb') as f: 
    pickle.dump(history, f)